# Pipeline Preditivo da MecâniQA

Entrega do encontro: integrar tratamento de valores nulos, padronização e modelo preditivo em um único `Pipeline` do Scikit-Learn. A divisão entre treino e teste respeita a ordem temporal dos dados para evitar vazamento de informações futuras.

## 1. Importação das bibliotecas

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## 2. Carregamento e organização temporal

A série principal continua sendo `Trocas_Oleo`, conforme a decisão já adotada pela equipe.

In [ ]:
data_path = next((
    path for path in [
        Path("data/mecaniqa_dataset.xlsx"),
        Path("../data/mecaniqa_dataset.xlsx"),
    ]
    if path.exists()
), None)

if data_path is None:
    raise FileNotFoundError(
        "Dataset não encontrado. Abra o projeto pela pasta mecaniQA-salvador."
    )

df = pd.read_excel(data_path)
df["Data"] = pd.to_datetime(df["Data"])
df = df.sort_values("Data").set_index("Data")

serie = df["Trocas_Oleo"].resample("D").asfreq()
serie.head()

## 3. Criação das variáveis preditoras

Todas as variáveis de histórico usam `shift(1)`: a previsão de um dia enxerga somente informações disponíveis até o dia anterior. Os valores nulos iniciais são mantidos para que o primeiro passo do pipeline os trate.

In [ ]:
dados_modelo = pd.DataFrame(index=serie.index)
dados_modelo["lag_1"] = serie.shift(1)
dados_modelo["lag_7"] = serie.shift(7)
dados_modelo["media_movel_7"] = serie.shift(1).rolling(7).mean()
dados_modelo["media_movel_30"] = serie.shift(1).rolling(30).mean()
dados_modelo["dia_semana"] = dados_modelo.index.dayofweek
dados_modelo["mes"] = dados_modelo.index.month
dados_modelo["alvo"] = serie

dados_modelo = dados_modelo.dropna(subset=["alvo"])
X = dados_modelo.drop(columns="alvo")
y = dados_modelo["alvo"]

X.shape, y.shape

## 4. Separação cronológica entre treino e teste

Os 20% finais da série ficam reservados para teste. Não há embaralhamento, pois isso misturaria passado e futuro.

In [ ]:
ponto_corte = int(len(X) * 0.80)
X_train, X_test = X.iloc[:ponto_corte], X.iloc[ponto_corte:]
y_train, y_test = y.iloc[:ponto_corte], y.iloc[ponto_corte:]

print(f"Treino: {X_train.index.min().date()} a {X_train.index.max().date()} ({len(X_train)} dias)")
print(f"Teste:  {X_test.index.min().date()} a {X_test.index.max().date()} ({len(X_test)} dias)")

## 5. Ajuste do hiperparâmetro do modelo

A regressão Ridge foi adotada porque trabalha bem com variáveis padronizadas e controla a complexidade por meio do hiperparâmetro `alpha`. A busca usa `TimeSeriesSplit`, preservando a ordem temporal em cada validação.

In [ ]:
pipeline_busca = Pipeline(steps=[
    ("imputacao", SimpleImputer(strategy="median")),
    ("padronizacao", StandardScaler()),
    ("modelo", Ridge())
])

busca = GridSearchCV(
    estimator=pipeline_busca,
    param_grid={"modelo__alpha": [0.01, 0.1, 1.0, 10.0, 100.0]},
    scoring="neg_mean_absolute_error",
    cv=TimeSeriesSplit(n_splits=5),
)
busca.fit(X_train, y_train)

melhor_alpha = busca.best_params_["modelo__alpha"]
print(f"Melhor alpha: {melhor_alpha}")

## 6. Pipeline final da equipe

A ordem segue o brainstorm: **preencher nulos → padronizar → aplicar o modelo preditivo tunado**. O `fit` é executado em uma única linha, como solicitado no guia.

In [ ]:
pipeline = Pipeline(steps=[
    ("imputacao", SimpleImputer(strategy="median")),
    ("padronizacao", StandardScaler()),
    ("modelo", Ridge(alpha=melhor_alpha))
])

pipeline.fit(X_train, y_train)

## 7. Avaliação no período de teste

In [ ]:
previsoes = pipeline.predict(X_test)

mae = mean_absolute_error(y_test, previsoes)
rmse = np.sqrt(mean_squared_error(y_test, previsoes))
r2 = r2_score(y_test, previsoes)

print(f"MAE:  {mae:.3f}")
print(f"RMSE: {rmse:.3f}")
print(f"R²:   {r2:.3f}")

In [ ]:
resultado = pd.DataFrame({
    "real": y_test,
    "previsao": previsoes,
}, index=y_test.index)

ax = resultado.plot(figsize=(14, 5), color=["#1f77b4", "#d62728"])
ax.set_title("Demanda real x previsão do Pipeline")
ax.set_xlabel("Data")
ax.set_ylabel("Trocas de óleo")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 8. Verificação contra vazamento de dados

Após o treinamento, o `SimpleImputer` e o `StandardScaler` guardam somente as estatísticas calculadas em `X_train`. Ao chamar `pipeline.predict(X_test)` ou prever dados novos, o pipeline executa apenas `transform` nesses dados; ele não recalcula mediana, média ou escala.